# Phase 1 — Explore raw federal book-purchase dumps

Índice de Bibliodiversidade. **This notebook is the review surface.**

- Reads compressed ZIPs from `SOURCE_DIR` (default Google Drive / `G:\O meu disco\...\Source`)
- Prints source resolution, inventory, zip members, parse choices, schema, dtypes, nulls, head rows, and NCM presence
- Column names are discovered at runtime — none are required
- **Does not** filter NCM, write parquet, or build Streamlit / Quarto / Actions


## 1. Engine


In [1]:
from __future__ import annotations

import io
import os
import zipfile
from pathlib import Path
from typing import Any

print("Índice de Bibliodiversidade — Phase 1 explore (notebook)")
print(f"cwd={Path.cwd()}")

try:
    import polars as pl
    print(f"engine=polars {pl.__version__}")
except ImportError:
    pl = None
    print("engine=polars MISSING — install with: pip install polars")

try:
    import duckdb
    print(f"fallback=duckdb {duckdb.__version__}")
except ImportError:
    duckdb = None
    print("fallback=duckdb not installed (ok)")


Índice de Bibliodiversidade — Phase 1 explore (notebook)
cwd=/workspace
engine=polars 1.44.2
fallback=duckdb not installed (ok)


## 2. Source resolution


In [2]:
DEFAULT_SOURCE_DIR = r"G:\O meu disco\Main\Business\TagusData\Projeto ComprasLivrosGov\Source"
RELATIVE_FALLBACKS = (Path("Source"), Path("data/raw"))
# Local copies already pulled on this VM (Drive MCP cannot download >10 MB).
VM_SAMPLES = (Path("/tmp/nfe-202609"), Path("/tmp/nfe-sample"))
TABULAR_SUFFIXES = (".csv", ".tsv", ".txt", ".parquet", ".pq")
ARCHIVE_SUFFIXES = (".zip",)
ENCODINGS = ("utf-8-sig", "utf-8", "latin-1", "cp1252")
SEPARATORS = (None, ";", ",", "\t", "|")
SAMPLE_FILES = 2
SAMPLE_ROWS = 50_000
HEAD_ROWS = 5
MEMBERS_PER_ZIP = 5


def linux_drive_aliases(win_path: str) -> list[Path]:
    cleaned = win_path.replace("\\", "/")
    if len(cleaned) >= 2 and cleaned[1] == ":":
        drive = cleaned[0].lower()
        rest = cleaned[2:].lstrip("/")
        return [
            Path(f"/mnt/{drive}") / rest,
            Path(f"/media/{drive}") / rest,
            Path("/mnt/g") / rest,
            Path("/mnt/google") / rest,
        ]
    return []


def candidate_source_dirs() -> list[Path]:
    out: list[Path] = []
    env = os.environ.get("SOURCE_DIR")
    if env:
        out.append(Path(env).expanduser())
    out.append(Path(DEFAULT_SOURCE_DIR))
    out.extend(linux_drive_aliases(DEFAULT_SOURCE_DIR))
    cwd = Path.cwd()
    for rel in RELATIVE_FALLBACKS:
        out.append(cwd / rel)
    out.extend(VM_SAMPLES)
    seen: set[str] = set()
    unique: list[Path] = []
    for p in out:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        unique.append(p)
    return unique


candidates = candidate_source_dirs()
print("SOURCE_DIR candidates (first existing directory wins):")
source_dir: Path | None = None
for path in candidates:
    exists = path.is_dir()
    print(f"  [{'OK' if exists else '  '}] {path}")
    if exists and source_dir is None:
        source_dir = path.resolve()

print()
if source_dir is None:
    print("RESULT: no SOURCE_DIR found. Place ZIPs in ./Source or data/raw, or set SOURCE_DIR.")
else:
    print(f"RESULT: SOURCE_DIR={source_dir}")
    if str(source_dir).startswith("/tmp/"):
        print("NOTE: using a local VM sample because G: / Drive is not mounted here.")


SOURCE_DIR candidates (first existing directory wins):
  [  ] G:\O meu disco\Main\Business\TagusData\Projeto ComprasLivrosGov\Source
  [  ] /mnt/g/O meu disco/Main/Business/TagusData/Projeto ComprasLivrosGov/Source
  [  ] /media/g/O meu disco/Main/Business/TagusData/Projeto ComprasLivrosGov/Source
  [  ] /mnt/google/O meu disco/Main/Business/TagusData/Projeto ComprasLivrosGov/Source
  [  ] /workspace/Source
  [  ] /workspace/data/raw
  [OK] /tmp/nfe-202609
  [OK] /tmp/nfe-sample

RESULT: SOURCE_DIR=/tmp/nfe-202609
NOTE: using a local VM sample because G: / Drive is not mounted here.


## 3. File inventory


In [3]:
def collect_raw_files(root: Path) -> list[Path]:
    files = [
        p
        for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in ARCHIVE_SUFFIXES + TABULAR_SUFFIXES
    ]
    files.sort(key=lambda p: (p.suffix.lower() != ".zip", p.name.lower()))
    return files


raw_files: list[Path] = []
if source_dir is None:
    print("skip inventory: no SOURCE_DIR")
else:
    raw_files = collect_raw_files(source_dir)
    zips = [p for p in raw_files if p.suffix.lower() == ".zip"]
    loose = [p for p in raw_files if p.suffix.lower() != ".zip"]
    print(f"inventory: {len(zips)} zip(s), {len(loose)} loose tabular file(s)")
    for p in raw_files[:50]:
        rel = p.relative_to(source_dir) if p.is_relative_to(source_dir) else p
        print(f"  - {rel}  ({p.stat().st_size} bytes)")
    if len(raw_files) > 50:
        print(f"  ... {len(raw_files) - 50} more")


inventory: 1 zip(s), 0 loose tabular file(s)
  - 202609_NFe.zip  (32969670 bytes)


## 4. Choose sample files (prefer recent large month)


In [4]:
def pick_sample(files: list[Path], n: int) -> list[Path]:
    zips = [p for p in files if p.suffix.lower() == ".zip"]
    # Prefer YYYYMM_NFe.zip with the newest year-month and ignore tiny stubs (< 1 MB).
    scored = []
    for p in zips:
        stem = p.stem  # 202609_NFe
        size = p.stat().st_size
        yyyymm = stem[:6] if len(stem) >= 6 and stem[:6].isdigit() else "000000"
        scored.append((size < 1_000_000, yyyymm, size, p))
    scored.sort(key=lambda t: (t[0], t[1], t[2]), reverse=True)
    chosen = [t[3] for t in scored[:n]]
    if not chosen:
        chosen = files[:n]
    return chosen


sampled = pick_sample(raw_files, SAMPLE_FILES) if raw_files else []
print(f"sample_files={len(sampled)}  (prefer newest large YYYYMM_NFe.zip)")
for p in sampled:
    print(f"  - {p.name}  ({p.stat().st_size} bytes)")


sample_files=1  (prefer newest large YYYYMM_NFe.zip)
  - 202609_NFe.zip  (32969670 bytes)


## 5. ZIP members


In [5]:
def is_tabular_name(name: str) -> bool:
    lower = name.lower()
    return any(lower.endswith(sfx) for sfx in TABULAR_SUFFIXES)


zip_members: dict[str, list[zipfile.ZipInfo]] = {}
if not sampled:
    print("skip members: no files sampled")
else:
    for path in sampled:
        if path.suffix.lower() != ".zip":
            print(f"LOOSE FILE {path.name} (not a zip)")
            continue
        print(f"######## ZIP {path.name} ########")
        print(f"path={path}")
        with zipfile.ZipFile(path) as zf:
            infos = [i for i in zf.infolist() if not i.is_dir()]
            zip_members[str(path)] = infos
            print(f"members={len(infos)}")
            for info in infos[:30]:
                kind = "tabular" if is_tabular_name(info.filename) else "other"
                print(f"  - {info.filename}  ({info.file_size} bytes, {kind})")
            if len(infos) > 30:
                print(f"  ... {len(infos) - 30} more")
        print()


######## ZIP 202609_NFe.zip ########
path=/tmp/nfe-202609/202609_NFe.zip
members=3
  - 202609_NFe_NotaFiscal.csv  (53325194 bytes, tabular)
  - 202609_NFe_NotaFiscalEvento.csv  (146 bytes, tabular)
  - 202609_NFe_NotaFiscalItem.csv  (217957440 bytes, tabular)



## 6. Parse helpers (encoding / separator probe)


In [6]:
def polars_read_csv(raw: bytes, encoding: str, sep: str | None, n_rows: int, infer: int):
    kwargs: dict[str, Any] = {
        "encoding": encoding,
        "n_rows": n_rows,
        "infer_schema_length": infer,
        "try_parse_dates": infer > 0,
        "ignore_errors": True,
    }
    if sep is not None:
        kwargs["separator"] = sep
    else:
        kwargs["separator"] = None
    return pl.read_csv(io.BytesIO(raw), **kwargs)


def annotate_inferred_dtypes(df_str, raw: bytes, encoding: str, sep: str | None, n_rows: int):
    try:
        df_inf = polars_read_csv(raw, encoding, sep, n_rows, infer=10_000)
    except Exception as exc:
        print(f"  infer-pass failed: {exc}")
        return {c: "String" for c in df_str.columns}, []
    notes: list[str] = []
    dtypes: dict[str, str] = {}
    for col in df_str.columns:
        as_str_nulls = int(df_str[col].null_count())
        if col not in df_inf.columns:
            dtypes[col] = "String"
            continue
        as_inf_nulls = int(df_inf[col].null_count())
        extra = as_inf_nulls - as_str_nulls
        if extra > 0:
            dtypes[col] = "String"
            notes.append(
                f"{col}: inferred {df_inf[col].dtype} added {extra} nulls "
                "(overflow/parse loss) — keep String"
            )
        else:
            dtypes[col] = str(df_inf[col].dtype)
    return dtypes, notes


def read_tabular_bytes(raw: bytes, name: str, n_rows: int):
    if pl is None:
        print("  cannot parse: polars not installed")
        return None, None
    lower = name.lower()
    if lower.endswith((".parquet", ".pq")):
        df = pl.read_parquet(io.BytesIO(raw)).head(n_rows)
        print("  parse: format=parquet")
        return df, {"format": "parquet"}

    last_err = None
    attempts = 0
    for encoding in ENCODINGS:
        for sep in SEPARATORS:
            attempts += 1
            try:
                df = polars_read_csv(raw, encoding, sep, n_rows, infer=0)
                if df.width == 0:
                    continue
                guessed, notes = annotate_inferred_dtypes(df, raw, encoding, sep, n_rows)
                sep_label = sep if sep is not None else "inferred"
                print(f"  parse: format=csv encoding={encoding} separator={sep_label!r} attempts={attempts}")
                return df, {
                    "format": "csv",
                    "encoding": encoding,
                    "separator": sep_label,
                    "inferred_dtypes": guessed,
                    "dtype_notes": notes,
                    "attempts": attempts,
                }
            except Exception as exc:
                last_err = exc
                continue
    print(f"  parse FAILED after {attempts} attempts: {last_err}")
    return None, None

print("helpers ready: encodings=", ENCODINGS, "separators=", SEPARATORS)


helpers ready: encodings= ('utf-8-sig', 'utf-8', 'latin-1', 'cp1252') separators= (None, ';', ',', '\t', '|')


## 7. Schema, dtypes, nulls, head rows


In [7]:
NCM_HINTS = ("ncm", "ncm/sh", "codigo ncm")


def print_profile(df, meta: dict[str, Any] | None, label: str) -> None:
    print(f"=== {label} ===")
    meta = dict(meta or {})
    inferred = meta.pop("inferred_dtypes", None)
    notes = meta.pop("dtype_notes", None)
    if meta:
        print("parse_meta:", ", ".join(f"{k}={v}" for k, v in meta.items()))
    cols = [str(c) for c in df.columns]
    print(f"rows_sampled={df.height}  ncols={len(cols)}")
    print("columns:")
    for i, c in enumerate(cols, 1):
        print(f"  {i:2d}. {c}")
    print("schema / dtypes:")
    for name, stored in df.schema.items():
        guess = inferred.get(str(name)) if inferred else None
        if guess and guess != str(stored):
            print(f"  - {name}: {stored}  (inferred {guess})")
        else:
            print(f"  - {name}: {stored}")
    if notes:
        print("dtype_notes:")
        for note in notes:
            print(f"  - {note}")
    print("null_counts (in sample):")
    nc = df.null_count().row(0)
    for name, count in zip(cols, nc):
        print(f"  - {name}: {int(count)}")
    print(f"first_rows (n={HEAD_ROWS}):")
    pl.Config.set_tbl_cols(-1)
    pl.Config.set_tbl_width_chars(200)
    pl.Config.set_fmt_str_lengths(80)
    print(df.head(HEAD_ROWS))


profiled = []
if not sampled:
    print("skip profile: no files sampled")
else:
    for path in sampled:
        if path.suffix.lower() == ".zip":
            with zipfile.ZipFile(path) as zf:
                infos = [i for i in zf.infolist() if not i.is_dir() and is_tabular_name(i.filename)]
                chosen = infos[:MEMBERS_PER_ZIP]
                print(f"profiling {len(chosen)} tabular member(s) from {path.name}")
                for info in chosen:
                    print()
                    print(f"-- member {info.filename} --")
                    raw = zf.read(info.filename)
                    print(f"  member_bytes={len(raw)}")
                    df, meta = read_tabular_bytes(raw, info.filename, SAMPLE_ROWS)
                    if df is None:
                        print("  unreadable as tabular")
                        continue
                    print_profile(df, meta, f"{path.name} :: {info.filename}")
                    profiled.append((path.name, info.filename, df, meta))
        else:
            print()
            print(f"-- file {path.name} --")
            raw = path.read_bytes()
            df, meta = read_tabular_bytes(raw, path.name, SAMPLE_ROWS)
            if df is None:
                print("  unreadable as tabular")
                continue
            print_profile(df, meta, str(path))
            profiled.append((path.name, path.name, df, meta))

print()
print(f"profiled_tables={len(profiled)}")


profiling 3 tabular member(s) from 202609_NFe.zip

-- member 202609_NFe_NotaFiscal.csv --


  member_bytes=53325194


  parse: format=csv encoding=latin-1 separator=';' attempts=12
=== 202609_NFe.zip :: 202609_NFe_NotaFiscal.csv ===
parse_meta: format=csv, encoding=latin-1, separator=;, attempts=12
rows_sampled=50000  ncols=25
columns:
   1. CHAVE DE ACESSO
   2. MODELO
   3. SÉRIE
   4. NÚMERO
   5. NATUREZA DA OPERAÇÃO
   6. DATA EMISSÃO
   7. EVENTO MAIS RECENTE
   8. DATA/HORA EVENTO MAIS RECENTE
   9. CPF/CNPJ Emitente
  10. RAZÃO SOCIAL EMITENTE
  11. INSCRIÇÃO ESTADUAL EMITENTE
  12. UF EMITENTE
  13. MUNICÍPIO EMITENTE
  14. CÓDIGO ÓRGÃO SUPERIOR DESTINATÁRIO
  15. ÓRGÃO SUPERIOR DESTINATÁRIO
  16. CÓDIGO ÓRGÃO DESTINATÁRIO
  17. ÓRGÃO DESTINATÁRIO
  18. CNPJ DESTINATÁRIO
  19. NOME DESTINATÁRIO
  20. UF DESTINATÁRIO
  21. INDICADOR IE DESTINATÁRIO
  22. DESTINO DA OPERAÇÃO
  23. CONSUMIDOR FINAL
  24. PRESENÇA DO COMPRADOR
  25. VALOR NOTA FISCAL
schema / dtypes:
  - CHAVE DE ACESSO: String
  - MODELO: String
  - SÉRIE: String  (inferred Int64)
  - NÚMERO: String  (inferred Int64)
  - NATUREZ

  member_bytes=217957440


  parse: format=csv encoding=latin-1 separator=';' attempts=12
=== 202609_NFe.zip :: 202609_NFe_NotaFiscalItem.csv ===
parse_meta: format=csv, encoding=latin-1, separator=;, attempts=12
rows_sampled=50000  ncols=31
columns:
   1. CHAVE DE ACESSO
   2. MODELO
   3. SÉRIE
   4. NÚMERO
   5. NATUREZA DA OPERAÇÃO
   6. DATA EMISSÃO
   7. CPF/CNPJ Emitente
   8. RAZÃO SOCIAL EMITENTE
   9. INSCRIÇÃO ESTADUAL EMITENTE
  10. UF EMITENTE
  11. MUNICÍPIO EMITENTE
  12. CÓDIGO ÓRGÃO SUPERIOR DESTINATÁRIO
  13. ÓRGÃO SUPERIOR DESTINATÁRIO
  14. CÓDIGO ÓRGÃO DESTINATÁRIO
  15. ÓRGÃO DESTINATÁRIO
  16. CNPJ DESTINATÁRIO
  17. NOME DESTINATÁRIO
  18. UF DESTINATÁRIO
  19. INDICADOR IE DESTINATÁRIO
  20. DESTINO DA OPERAÇÃO
  21. CONSUMIDOR FINAL
  22. PRESENÇA DO COMPRADOR
  23. NÚMERO PRODUTO
  24. DESCRIÇÃO DO PRODUTO/SERVIÇO
  25. CÓDIGO NCM/SH
  26. NCM/SH (TIPO DE PRODUTO)
  27. CFOP
  28. QUANTIDADE
  29. UNIDADE
  30. VALOR UNITÁRIO
  31. VALOR TOTAL
schema / dtypes:
  - CHAVE DE ACESSO: Stri

## 8. NCM presence (no filter)


In [8]:
print("NCM presence check — does NOT filter rows or write parquet.")
print("Looking for a column whose name contains 'ncm' (case-insensitive).")

if not profiled:
    print("no profiled tables")
else:
    for zip_name, member, df, _meta in profiled:
        cols = [str(c) for c in df.columns]
        ncm_cols = [c for c in cols if "ncm" in c.lower()]
        print()
        print(f"{zip_name} :: {member}")
        if not ncm_cols:
            print("  NCM column: ABSENT")
            continue
        print(f"  NCM column(s): {ncm_cols}")
        for col in ncm_cols:
            s = df[col]
            nulls = int(s.null_count())
            empty = int((s.cast(pl.String) == "").sum()) if s.dtype == pl.String else 0
            print(f"  {col}: nulls={nulls} empty={empty} unique_in_sample={s.n_unique()}")
            print(f"    sample={s.drop_nulls().head(8).to_list()}")
            as_str = s.cast(pl.String)
            bookish = as_str.str.contains(r"^490")
            n_book = int(bookish.sum())
            print(f"    values starting with 490 (book chapter, sample only): {n_book}")
            if n_book:
                print(f"    examples={as_str.filter(bookish).unique().head(10).to_list()}")
            desc_cols = [c for c in cols if "ncm" in c.lower() and c != col]
            for dcol in desc_cols:
                paired = (
                    df.filter(bookish)
                    .select([col, dcol])
                    .unique()
                    .head(8)
                )
                if paired.height:
                    print(f"    {dcol} for 490* in sample:")
                    print(paired)

print()
print("Done. Phase 1 only — no NCM filter, no parquet, no Streamlit/Quarto/Actions.")


NCM presence check — does NOT filter rows or write parquet.
Looking for a column whose name contains 'ncm' (case-insensitive).

202609_NFe.zip :: 202609_NFe_NotaFiscal.csv
  NCM column: ABSENT

202609_NFe.zip :: 202609_NFe_NotaFiscalEvento.csv
  NCM column: ABSENT

202609_NFe.zip :: 202609_NFe_NotaFiscalItem.csv
  NCM column(s): ['CÓDIGO NCM/SH', 'NCM/SH (TIPO DE PRODUTO)']
  CÓDIGO NCM/SH: nulls=0 empty=0 unique_in_sample=2643
    sample=['4012010', '4012010', '68138110', '84213990', '87089990', '87083090', '27101932', '38140090']
    values starting with 490 (book chapter, sample only): 833
    examples=['49059000', '49011000', '49029000', '49019900', '49089000', '49021000', '49070010', '49090000']
    NCM/SH (TIPO DE PRODUTO) for 490* in sample:
shape: (8, 2)
┌───────────────┬───────────────────────────────────────────────────────────────────────────────────┐
│ CÓDIGO NCM/SH ┆ NCM/SH (TIPO DE PRODUTO)                                                          │
│ ---           ┆ ---  